In [1]:
import pandas as pd
import matplotlib.pyplot as plt

In [9]:
results = (pd.read_pickle('../sim/data/simulation_results_20260225_211042.pkl'))

In [ ]:
# Add losses column
df['losses'] = 1 - df['won']  # since won=1, loss=0
epsilon = 1

# Group by run + rep_strategy + account_personality
grouped = df.groupby(["run", "rep_strategy", "account_personality"]).agg(
    wins=("won", "sum"),
    losses=("losses", "sum"),
    total=("won", "count")
).reset_index()

# Compute per-run win/loss ratio
grouped["win_loss_ratio"] = grouped["wins"] / (grouped["losses"] + epsilon)

# Now average across runs for each rep × personality combo
avg_ratio = grouped.groupby(["rep_strategy", "account_personality"])["win_loss_ratio"].mean().reset_index()

# Pivot for heatmap
heatmap_matrix = avg_ratio.pivot(
    index="rep_strategy",
    columns="account_personality",
    values="win_loss_ratio"
)

# Optional: counts (average total opps per run)
counts_matrix = grouped.groupby(["rep_strategy", "account_personality"])["total"].mean().unstack()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
cax = ax.imshow(heatmap_matrix, cmap='coolwarm', vmin=1, vmax=np.nanmax(heatmap_matrix))

ax.set_xticks(np.arange(len(heatmap_matrix.columns)))
ax.set_yticks(np.arange(len(heatmap_matrix.index)))
ax.set_xticklabels(heatmap_matrix.columns)
ax.set_yticklabels(heatmap_matrix.index)
plt.setp(ax.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor")

# Annotate with ratio and average counts
for i, rep in enumerate(heatmap_matrix.index):
    for j, acc in enumerate(heatmap_matrix.columns):
        value = heatmap_matrix.iloc[i, j]
        count = counts_matrix.iloc[i, j]
        if not np.isnan(value):
            ax.text(j, i, f"{value:.2f}\n({count:.0f})", ha="center", va="center", color="black")
        else:
            ax.text(j, i, "n/a", ha="center", va="center", color="red")

cbar = fig.colorbar(cax, ax=ax)
cbar.set_label('Win / Loss Ratio (averaged across runs)')
ax.set_title("Rep Strategy × Account Personality Win/Loss Ratio (Average per Run)")
plt.tight_layout()
plt.savefig('heatmap.png')
plt.show()